# Stage 3 — Spark SQL Business Insights

This notebook is **Stage 3** of the split pipeline. It contains the original Spark SQL section (Sec. 17: SQL 1–10, each with its own visualization) copied over **unchanged**.

It depends on Stage 1's output (`df_clean.parquet`, `session_dataset.parquet`), loaded in the setup cell below and registered as the same `events` / `sessions` temp views the original queries expect.

**Requires:** `df_clean.parquet`, `session_dataset.parquet` (from Stage 1)

In [ ]:
# --- Added for orchestration: scoped dependency install ---
# Revised from the original notebook's single install cell (which
# installed pyarrow + xgboost[spark] together for the whole notebook).
# This notebook only needs: pyarrow (fast Arrow transfer for the .toPandas() calls behind each SQL query's visualization). xgboost is NOT needed here -- no ML in this notebook.
import subprocess
import sys

packages = [
    'pyarrow'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])
    print(pkg, 'ready')

print('All dependencies installed.')

In [ ]:
# --- Added for orchestration: Spark session + load Stage 1 output ---
import os
DATA_DIR = os.environ.get("ECOMM_DATA_DIR", "/tmp/ecommerce_pipeline/data")
os.makedirs(DATA_DIR, exist_ok=True)
print("Using DATA_DIR:", DATA_DIR)

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100

spark = (
    SparkSession.builder
    .appName("SQL-Business-Insights")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")
spark.sparkContext.setLogLevel("WARN")

df_clean = spark.read.parquet(f"{DATA_DIR}/df_clean.parquet")
session_dataset = spark.read.parquet(f"{DATA_DIR}/session_dataset.parquet")
print("Loaded df_clean:", df_clean.count(), "rows")
print("Loaded session_dataset:", session_dataset.count(), "rows")

---
## 17. Spark SQL — Business Insights

The pipeline above uses the Spark DataFrame API throughout. Spark SQL is equally valid on the same
underlying DataFrames, so this section registers `df_clean` (event-level, cleaned) and `session_dataset`
(session-level, from Section 9.2) as temp views and answers a set of business questions directly in SQL.
This mixes plain `SELECT`/`GROUP BY` aggregations, a `HAVING` filter, a `CASE WHEN` funnel query, a
`self-JOIN`, a `CTE` (`WITH`), and a `WINDOW` function query.

In [ ]:
# Register temp views so we can query with spark.sql(...)
df_clean.createOrReplaceTempView("events")
session_dataset.createOrReplaceTempView("sessions")

### SQL 1 — Event Type Counts (plain `SELECT` + `GROUP BY`)

In [ ]:
spark.sql("""
    SELECT event_type, COUNT(*) AS n_events
    FROM events
    GROUP BY event_type
    ORDER BY n_events DESC
""").show()

### SQL 2 — View → Cart → Purchase Funnel per Category (`CASE WHEN` pivot)

In [ ]:
spark.sql("""
    SELECT
        category_code,
        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
        ROUND(
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) * 100.0
            / NULLIF(SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END), 0),
            4
        ) AS view_to_purchase_pct
    FROM events
    WHERE category_code != 'Unknown'
    GROUP BY category_code
    ORDER BY purchases DESC
    LIMIT 15
""").show(truncate=False)

In [ ]:
# SQL 2 Visualization: Category Conversion Funnel & View-to-Purchase Pct
sql2_df = spark.sql("""
    SELECT
        category_code,
        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
        ROUND(
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) * 100.0
            / NULLIF(SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END), 0),
            4
        ) AS view_to_purchase_pct
    FROM events
    WHERE category_code != 'Unknown'
    GROUP BY category_code
    ORDER BY purchases DESC
    LIMIT 10
""")
sql2_pd = sql2_df.toPandas()

import matplotlib.pyplot as plt
import numpy as np

fig, ax1 = plt.subplots(figsize=(12, 6))
x = np.arange(len(sql2_pd["category_code"]))
width = 0.25

ax1.bar(x - width, sql2_pd["views"], width, label="Views", color="#4C72B0")
ax1.bar(x, sql2_pd["carts"], width, label="Carts", color="#DD8452")
ax1.bar(x + width, sql2_pd["purchases"], width, label="Purchases", color="#55A868")
ax1.set_ylabel("Event Volume", fontsize=11)
ax1.set_title("SQL Query 2: Conversion Funnel & Purchase Rate across Categories", fontsize=13, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(sql2_pd["category_code"], rotation=45, ha="right", fontsize=9)
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(x, sql2_pd["view_to_purchase_pct"], color="#C44E52", marker="o", linewidth=2.5, label="Conversion Rate (%)")
ax2.set_ylabel("Conversion Rate (%)", color="#C44E52", fontsize=11)
ax2.tick_params(axis="y", labelcolor="#C44E52")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig("sql_plot_01_category_funnel.png", dpi=150)
plt.show()


### SQL 3 — Top Brands by Purchase Revenue (`WHERE` + `GROUP BY` + `ORDER BY`)

In [ ]:
spark.sql("""
    SELECT brand, ROUND(SUM(price), 2) AS total_revenue, COUNT(*) AS n_purchases
    FROM events
    WHERE event_type = 'purchase' AND brand != 'Unknown'
    GROUP BY brand
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# SQL 3 Visualization: Top Brands by Purchase Revenue
sql3_df = spark.sql("""
    SELECT brand, ROUND(SUM(price), 2) AS total_revenue, COUNT(*) AS n_purchases
    FROM events
    WHERE event_type = 'purchase' AND brand != 'Unknown'
    GROUP BY brand
    ORDER BY total_revenue DESC
    LIMIT 10
""")
sql3_pd = sql3_df.toPandas()

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(sql3_pd["brand"][::-1], sql3_pd["total_revenue"][::-1] / 1e6, color="#4C72B0")
ax.set_xlabel("Total Revenue ($ Millions)", fontsize=11)
ax.set_ylabel("Brand", fontsize=11)
ax.set_title("SQL Query 3: Top 10 Revenue-Generating Brands", fontsize=13, fontweight="bold")

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.05, bar.get_y() + bar.get_height()/2, f" ${w:.2f}M", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("sql_plot_02_top_brands_revenue.png", dpi=150)
plt.show()


### SQL 4 — Revenue Rank per Brand within Category (`WINDOW` function)

In [ ]:
spark.sql("""
    -- SQL 4: Revenue rank per brand within category (WINDOW function)
    -- Note: QUALIFY is replaced with a subquery for Spark 3.x compatibility.
    SELECT category_code, brand, ROUND(revenue, 2) AS revenue, revenue_rank
    FROM (
        SELECT
            category_code,
            brand,
            SUM(price) AS revenue,
            RANK() OVER (PARTITION BY category_code ORDER BY SUM(price) DESC) AS revenue_rank
        FROM events
        WHERE event_type = 'purchase'
          AND category_code != 'Unknown'
          AND brand != 'Unknown'
        GROUP BY category_code, brand
    ) ranked
    WHERE revenue_rank <= 2
    ORDER BY category_code, revenue_rank
""").show(30, truncate=False)


*(If your Spark version doesn't support `QUALIFY`, the equivalent is wrapping the `WINDOW` query in a
subquery and filtering `WHERE revenue_rank <= 2` in the outer `SELECT` — Spark 3.3, as used in this
notebook's `SparkSession`, supports `QUALIFY` natively.)*

### SQL 5 — Purchases by Hour of Day (`GROUP BY` on a derived column)

In [ ]:
spark.sql("""
    SELECT HOUR(event_time_ts) AS hour_of_day, COUNT(*) AS n_purchases
    FROM events
    WHERE event_type = 'purchase'
    GROUP BY HOUR(event_time_ts)
    ORDER BY hour_of_day
""").show(24)

In [ ]:
# SQL 5 Visualization: Hourly Purchase Pattern
sql5_df = spark.sql("""
    SELECT HOUR(event_time_ts) AS hour_of_day, COUNT(*) AS n_purchases
    FROM events
    WHERE event_type = 'purchase'
    GROUP BY HOUR(event_time_ts)
    ORDER BY hour_of_day
""")
sql5_pd = sql5_df.toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sql5_pd["hour_of_day"], sql5_pd["n_purchases"], color="#55A868", marker="o", linewidth=2.5)
ax.fill_between(sql5_pd["hour_of_day"], sql5_pd["n_purchases"], color="#55A868", alpha=0.2)
ax.set_xticks(range(0, 24))
ax.set_xlabel("Hour of Day (UTC)", fontsize=11)
ax.set_ylabel("Total Purchases", fontsize=11)
ax.set_title("SQL Query 5: Hourly Distribution of Purchases Throughout the Day", fontsize=13, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("sql_plot_03_hourly_purchases.png", dpi=150)
plt.show()


### SQL 6 — High-Interest, Low-Conversion Products (`GROUP BY` + `HAVING`)

In [ ]:
spark.sql("""
    SELECT
        product_id,
        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases
    FROM events
    GROUP BY product_id
    HAVING views > 500 AND purchases = 0
    ORDER BY views DESC
    LIMIT 10
""").show(truncate=False)

### SQL 7 — Users with Multiple Sessions (`COUNT(DISTINCT ...)` + `HAVING`)

In [ ]:
spark.sql("""
    SELECT user_id, COUNT(DISTINCT user_session) AS n_sessions
    FROM events
    GROUP BY user_id
    HAVING COUNT(DISTINCT user_session) > 1
    ORDER BY n_sessions DESC
    LIMIT 10
""").show(truncate=False)

### SQL 8 — Session Outcome via `SUM() OVER (PARTITION BY ...)` (window alternative to the Section 9.2 `groupBy`)

In [ ]:
spark.sql("""
    SELECT DISTINCT
        user_session,
        SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END)
            OVER (PARTITION BY user_session) AS session_total_spending_sql,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
            OVER (PARTITION BY user_session) AS will_purchase_sql
    FROM events
    ORDER BY session_total_spending_sql DESC
    LIMIT 10
""").show(truncate=False)

### SQL 9 — Average Spend per Purchasing Session, by Day of Week (query on `sessions` view)

In [ ]:
spark.sql("""
    SELECT
        session_dayofweek,
        COUNT(*) AS n_purchasing_sessions,
        ROUND(AVG(session_total_spending), 2) AS avg_spend
    FROM sessions
    WHERE will_purchase = 1
    GROUP BY session_dayofweek
    ORDER BY session_dayofweek
""").show()

### SQL 10 — Self-`JOIN`: Brands Frequently Co-Viewed with Apple Products in the Same Session

In [ ]:
spark.sql("""
    SELECT e2.brand AS co_viewed_brand, COUNT(*) AS co_view_count
    FROM events e1
    JOIN events e2
      ON e1.user_session = e2.user_session
     AND e1.brand = 'apple'
     AND e2.brand != 'apple'
     AND e2.brand != 'Unknown'
    WHERE e1.event_type = 'view' AND e2.event_type = 'view'
    GROUP BY e2.brand
    ORDER BY co_view_count DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# SQL 10 Visualization: Brands Frequently Co-Viewed with Apple (Self-JOIN)
sql10_df = spark.sql("""
    SELECT e2.brand AS co_viewed_brand, COUNT(*) AS co_view_count
    FROM events e1
    JOIN events e2
      ON e1.user_session = e2.user_session
     AND e1.brand = 'apple'
     AND e2.brand != 'apple'
     AND e2.brand != 'Unknown'
    WHERE e1.event_type = 'view' AND e2.event_type = 'view'
    GROUP BY e2.brand
    ORDER BY co_view_count DESC
    LIMIT 10
""")
sql10_pd = sql10_df.toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(sql10_pd["co_viewed_brand"], sql10_pd["co_view_count"], color="#DD8452")
ax.set_xlabel("Co-Viewed Brand", fontsize=11)
ax.set_ylabel("Co-View Count in Apple Sessions", fontsize=11)
ax.set_title("SQL Query 10: Top Brands Co-Viewed in Apple Sessions (Self-JOIN)", fontsize=13, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9)

for i, v in enumerate(sql10_pd["co_view_count"]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8.5)

plt.tight_layout()
plt.savefig("sql_plot_04_apple_coviewed_brands.png", dpi=150)
plt.show()
